# Evaluation SRGAN

This source code is used to **evaluate the performance of the SRGAN model** by calculating **Mean Squared Error (MSE)**, **Peak Signal-to-Noise Ratio (PSNR)**, and **Structural Similarity Index Measure (SSIM)**. The evaluation is performed by comparing the generated **Super-Resolution (SR)** images with the corresponding **Low-Resolution (LR)** input images.

---

# 1. Import library

In [ ]:
import os
import sys
import csv
import glob
import importlib
import math
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.ndimage import shift as nd_shift
from skimage.color import rgb2gray
from skimage.metrics import (
    mean_squared_error as mse_func,
    peak_signal_noise_ratio as psnr_func,
    structural_similarity as ssim_func,
)
from skimage.metrics import structural_similarity as ssim
from skimage.registration import phase_cross_correlation
from torchvision.transforms import ToPILImage, ToTensor
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 2. Setup

### 2.1. PATH

In [ ]:
GENERATOR_PATH = "utils"
MODEL_PATH = "result"
OUTPUT_PATH = "result"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CSV_PATH = f"{OUTPUT_PATH}/Evaluation_result.csv"

In [ ]:
base_ref = "dataset"
base_test = "dataset_SRGAN"

### 2.2 Config

Adjust the **HR_SIZE**

In [ ]:
HR_SIZE = 256
SCALE = 4
GLOBAL_RESIZE = None

NUM_PAIRS = 5 #Number of Example pairs
IMAGE_PAIRS = []

# 3. Utils

### 3.1. Example Pairs Preparation

In [ ]:
IMG_EXTS = {".png", ".jpg", ".jpeg"}

def pick_image(base_dir, fold_dir, class_name, index):
    folder = os.path.join(base_dir, fold_dir, class_name)

    files = sorted([
        f for f in glob.glob(os.path.join(folder, "*"))
        if os.path.splitext(f)[1].lower() in IMG_EXTS
    ])

    if not files:
        raise FileNotFoundError(f"No images found in '{folder}'")

    if index >= len(files):
        raise IndexError(
            f"index={index} exceeds number of files ({len(files)}) in '{folder}'"
        )

    return files[index]

In [ ]:
for i in range(NUM_PAIRS):
    fold_dir = f"fold_1"
    index = random.randint(1, 1000)

    ref_path = pick_image(
        base_ref,
        fold_dir,
        CLASS_NAMEE,
        index
    )

    test_path = pick_image(
        base_test,
        fold_dir,
        CLASS_NAME,
        index
    )

    IMAGE_PAIRS.append({
        "name": f"pair_{i+1:02d}",
        "path_ref": ref_path,
        "path_test": test_path
    })

for pair in IMAGE_PAIRS:
    print(pair)

In [ ]:
def load_image_from_path(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'File Not Found: {path}')
    img = Image.open(path).convert('RGB')

    return np.array(img)

def resize_img(img, size):
    return cv2.resize(img, size, interpolation=cv2.INTER_CUBIC)

def rgb2ycbcr_y_channel(img):
    img_255 = img * 255.0
    # RGB order: [R, G, B] = [65.481, 128.553, 24.966]
    y_channel = np.dot(img_255, [65.481, 128.553, 24.966]) / 255.0 + 16.0
    return y_channel

In [ ]:
def compute_metrics(img_ref, img_test, crop_border=0):
    if img_ref.dtype != np.float32:
        img_ref = img_ref.astype(np.float32) / 255.0
    if img_test.dtype != np.float32:
        img_test = img_test.astype(np.float32) / 255.0

    h, w = img_ref.shape[:2]
    img_test = resize_img(img_test, (w, h))
    img_test = np.clip(img_test, 0.0, 1.0)

    if crop_border > 0:
        img_ref  = img_ref [crop_border:-crop_border, crop_border:-crop_border]
        img_test = img_test[crop_border:-crop_border, crop_border:-crop_border]

    if img_ref.ndim == 3:
        img_ref  = rgb2ycbcr_y_channel(img_ref)
        img_test = rgb2ycbcr_y_channel(img_test)
    else:
        img_ref  = img_ref  * 255.0
        img_test = img_test * 255.0

    # MSE
    mse_val = np.mean((img_ref - img_test) ** 2)

    # PSNR
    psnr_val = float("inf") if mse_val == 0 else 20 * math.log10(255.0 / math.sqrt(mse_val))

    # SSIM
    ssim_val = ssim(
        img_ref, img_test,
        data_range=255.0,
        gaussian_weights=True,
        sigma=1.5,
        use_sample_covariance=False,
    )

    return {"mse": float(mse_val), "psnr": float(psnr_val), "ssim": float(ssim_val)}

### 3.2. Load SRGAN

In [ ]:
def load_generator(model_path):
    if GENERATOR_PATH not in sys.path:
        sys.path.insert(0, GENERATOR_PATH)

    module = importlib.import_module("model")
    importlib.reload(module)
    Generator = module.Generator
    print(f"Loaded model file: {GENERATOR_PATH}/model.py")

    generator = Generator(SCALE).to(device)

    weight_path = f"{model_path}/model.pth"
    state_dict = torch.load(weight_path, map_location=device)

    missing, unexpected = generator.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"Missing keys: {missing}")
    if unexpected:
        print(f"Unexpected keys: {unexpected}")
    if not missing and not unexpected:
        print("All weights loaded correctly")

    generator.eval()
    print("Model loaded successfully.")
    return generator

In [ ]:
def enhance_with_srgan(input_path, generator):
    img = cv2.imread(input_path)
    if img is None:
        raise FileNotFoundError(f"File Not Found: {input_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lr_tensor = ToTensor()(img_rgb).unsqueeze(0).to(device)

    with torch.no_grad():
        sr_tensor = generator(lr_tensor)

    sr_tensor = sr_tensor.squeeze(0).cpu()
    if sr_tensor.min() < 0:
        sr_tensor = (sr_tensor + 1) / 2
    sr_tensor = sr_tensor.clamp(0, 1)

    img_sr = np.array(ToPILImage()(sr_tensor))
    img_sr = np.clip(img_sr, 0, 255).astype(np.uint8)

    img_sr_yuv = cv2.cvtColor(img_sr,    cv2.COLOR_RGB2YCrCb).astype(np.float32)
    img_lr_yuv = cv2.cvtColor(img_rgb,   cv2.COLOR_RGB2YCrCb).astype(np.float32)

    sr_y = img_sr_yuv[..., 0]
    lr_y = img_lr_yuv[..., 0]

    sr_y_matched = (sr_y - sr_y.mean()) / (sr_y.std() + 1e-6) \
                   * lr_y.std() + lr_y.mean()

    img_sr_yuv[..., 0] = np.clip(sr_y_matched, 0, 255)
    img_sr = cv2.cvtColor(img_sr_yuv.astype(np.uint8), cv2.COLOR_YCrCb2RGB)

    return img_sr

### 3.3. Evaluation Process

In [ ]:
generator = load_generator(MODEL_PATH)

def evaluate_all_pairs(path_lr, path_hr, pair_name=None):
    name = pair_name or "pair"
    img_hr = load_image_from_path(path_hr)
    img_lr = load_image_from_path(path_lr)

    h_hr, w_hr = img_hr.shape[:2]
    hr_size = (w_hr, h_hr)

    # ---------- LR ------------
    img_lr_up = cv2.resize(img_lr, hr_size, interpolation=cv2.INTER_NEAREST)
    m_lr = compute_metrics(img_hr, img_lr_up, crop_border=8)
    print(f" LR  vs HR (Y) → MSE={m_lr['mse']:.4f}  PSNR={m_lr['psnr']:.4f} dB  SSIM={m_lr['ssim']:.4f}")

    # ---------- SRGAN ----------
    img_sr = enhance_with_srgan(path_lr, generator)
    m_sr = compute_metrics(img_hr, img_sr, crop_border=8)
    print(f"SRGAN vs HR (Y) → MSE={m_sr['mse']:.4f}  PSNR={m_sr['psnr']:.4f} dB  SSIM={m_sr['ssim']:.4f}")

    return {
        "pair_name": name,
        "lr_mse":  round(m_lr["mse"],  4),
        "lr_psnr": round(m_lr["psnr"], 4),
        "lr_ssim": round(m_lr["ssim"], 4),
        "sr_mse":  round(m_sr["mse"],  4),
        "sr_psnr": round(m_sr["psnr"], 4),
        "sr_ssim": round(m_sr["ssim"], 4),
    }

In [ ]:
def evaluate_from_list(image_pairs):
    results = []

    for pair in image_pairs:
        name = pair["name"]
        ref = pair["path_ref"]
        test = pair["path_test"]

        result = evaluate_all_pairs(test, ref, pair_name=name)
        results.append(result)

    return results

### 3.4. CSV File Structure

In [ ]:
def _stat_rows(results: list, key_prefix: str) -> tuple:
    valid = [r for r in results if f'{key_prefix}mse' in r]
    if not valid:
        return {}, {}
    avg = {
        f'{key_prefix}mse':  round(np.mean([r[f'{key_prefix}mse']  for r in valid]), 4),
        f'{key_prefix}psnr': round(np.mean([r[f'{key_prefix}psnr'] for r in valid]), 4),
        f'{key_prefix}ssim': round(np.mean([r[f'{key_prefix}ssim'] for r in valid]), 4),
    }
    sd = (
        {
            f'{key_prefix}mse':  round(np.std([r[f'{key_prefix}mse']  for r in valid], ddof=1), 4),
            f'{key_prefix}psnr': round(np.std([r[f'{key_prefix}psnr'] for r in valid], ddof=1), 4),
            f'{key_prefix}ssim': round(np.std([r[f'{key_prefix}ssim'] for r in valid], ddof=1), 4),
        }
        if len(valid) > 1 else
        {f'{key_prefix}mse': 0, f'{key_prefix}psnr': 0, f'{key_prefix}ssim': 0}
    )
    return avg, sd

In [ ]:
def print_summary(results: list):
    print("\n===== Evaluation Summary =====\n")

    valid = [r for r in results if 'lr_mse' in r]

    for r in results:
        if 'lr_mse' not in r:
            print(f"{r['pair_name']} : ERROR (missing metrics)")
            continue

        print(f"Pair: {r['pair_name']}")
        print(f"  LR vs HR  -  MSE: {r['lr_mse']}   PSNR: {r['lr_psnr']}   SSIM: {r['lr_ssim']}")
        print(f"  SR vs HR  -  MSE: {r['sr_mse']}   PSNR: {r['sr_psnr']}   SSIM: {r['sr_ssim']}")
        print()

    if len(valid) > 0:
        print("===== AVERAGE =====")
        print(f"LR MSE : {np.mean([r['lr_mse'] for r in valid])}")
        print(f"LR PSNR: {np.mean([r['lr_psnr'] for r in valid])}")
        print(f"LR SSIM: {np.mean([r['lr_ssim'] for r in valid])}")

        print(f"SR MSE : {np.mean([r['sr_mse'] for r in valid])}")
        print(f"SR PSNR: {np.mean([r['sr_psnr'] for r in valid])}")
        print(f"SR SSIM: {np.mean([r['sr_ssim'] for r in valid])}")

        print("\n===== STD (Standard Deviation) =====")
        print(f"LR MSE : {np.std([r['lr_mse'] for r in valid], ddof=1)}")
        print(f"LR PSNR: {np.std([r['lr_psnr'] for r in valid], ddof=1)}")
        print(f"LR SSIM: {np.std([r['lr_ssim'] for r in valid], ddof=1)}")

        print(f"SR MSE : {np.std([r['sr_mse'] for r in valid], ddof=1)}")
        print(f"SR PSNR: {np.std([r['sr_psnr'] for r in valid], ddof=1)}")
        print(f"SR SSIM: {np.std([r['sr_ssim'] for r in valid], ddof=1)}")

    print("\n===== END =====\n")

In [ ]:
def save_to_csv(results, csv_path):

    fields = [
        'pair_name',
        'lr_mse', 'lr_psnr', 'lr_ssim',
        'sr_mse', 'sr_psnr', 'sr_ssim',
    ]
    valid = [r for r in results if 'lr_mse' in r]
    avg_lr, sd_lr = _stat_rows(valid, 'lr_')
    avg_sr, sd_sr = _stat_rows(valid, 'sr_')

    avg_row = {'pair_name': 'AVERAGE', **avg_lr, **avg_sr}
    sd_row  = {'pair_name': 'SD', **sd_lr, **sd_sr}

    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(results)
        writer.writerow({k: '' for k in fields})
        writer.writerow(avg_row)
        writer.writerow(sd_row)

    print(f'\n Save to CSV: {os.path.abspath(csv_path)}')
    return csv_path

# 4. SRGAN Evaluation

In [ ]:
results = evaluate_from_list(IMAGE_PAIRS)
print_summary(results)
save_to_csv(results, CSV_PATH)

df = pd.read_csv(CSV_PATH)
display(df)